# Session 4: 모델 학습과 교차 검증

**DevOps 엔지니어를 위한 ML 입문 - 4/5회차**

---

## 학습 목표

1. 머신러닝 모델의 종류와 특성을 이해한다
2. 과적합과 과소적합의 개념을 DevOps 관점에서 이해한다
3. K-Fold 교차 검증으로 신뢰할 수 있는 성능 측정을 수행한다
4. 하이퍼파라미터 튜닝을 통해 모델 성능을 개선한다
5. 최종 제출 파일을 생성한다

---

# Part 1: 이론 (30분)

## 1.1 모델 종류 (간단 비교)

머신러닝 모델은 크게 세 가지 계열로 나눌 수 있습니다.

### 선형 모델 (Ridge Regression)

| 항목 | 설명 |
|------|------|
| **원리** | 입력 피처들의 가중합으로 예측값을 계산 |
| **장점** | 단순하고 빠르며 해석이 쉬움 |
| **단점** | 비선형 관계를 포착하지 못함 |
| **DevOps 비유** | 로드밸런서의 가중치 기반 라우팅 - 각 서버에 가중치를 부여해서 트래픽을 분배하는 것처럼, 각 피처에 가중치를 곱해서 결과를 계산 |

### 트리 기반 모델 (Random Forest)

| 항목 | 설명 |
|------|------|
| **원리** | 여러 개의 결정 트리를 만들어 평균/다수결로 예측 |
| **장점** | 비선형 관계 포착 가능, 과적합에 강함 |
| **단점** | 느리고, 학습 데이터 범위 밖 예측이 어려움 |
| **DevOps 비유** | 마이크로서비스 아키텍처 - 여러 독립적인 서비스(트리)가 각각 판단하고, 최종 결과를 종합하는 API Gateway(앙상블)가 있는 구조 |

### 부스팅 모델 (LightGBM)

| 항목 | 설명 |
|------|------|
| **원리** | 이전 트리의 오차를 다음 트리가 보정하며 순차적으로 학습 |
| **장점** | 높은 성능, 빠른 학습 속도, 현업에서 가장 많이 사용 |
| **단점** | 하이퍼파라미터 튜닝이 중요, 과적합 위험 |
| **DevOps 비유** | CI/CD 파이프라인 - 각 스테이지(트리)가 이전 스테이지의 실패를 보완하며 점진적으로 품질을 높이는 방식 |

---

> **현업 TIP**: Kaggle 대회와 실무에서 정형 데이터(테이블 데이터)에는 LightGBM, XGBoost 같은 부스팅 모델이 가장 많이 사용됩니다. 딥러닝은 이미지, 텍스트 같은 비정형 데이터에 주로 사용합니다.

## 1.2 과적합 vs 과소적합

### 과적합 (Overfitting)

모델이 학습 데이터의 노이즈까지 외워버려서 새로운 데이터에 대한 예측 성능이 떨어지는 현상입니다.

**DevOps 비유: 모니터링 알람이 너무 민감한 상태**
- CPU 사용률이 0.1%만 올라가도 알람이 울림
- 학습 데이터의 사소한 패턴까지 모두 반응
- 정작 중요한 장애와 일반적인 변동을 구분하지 못함

### 과소적합 (Underfitting)

모델이 데이터의 패턴을 충분히 학습하지 못한 상태입니다.

**DevOps 비유: 모니터링 알람이 너무 둔감한 상태**
- CPU가 90%에 도달해도 알람이 울리지 않음
- 중요한 패턴을 놓치고 있음
- 모델 복잡도가 너무 낮거나, 학습이 부족한 상태

### 적절한 균형점 찾기

<div style="margin: 20px 0; max-width: 650px;">
  <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
    <div style="text-align: center; flex: 1;">
      <div style="border: 2px solid #3498db; border-radius: 8px; padding: 10px; background: #ebf5fb;">
        <div style="font-weight: bold; color: #2980b9;">과소적합</div>
        <div style="font-size: 0.8em; color: #555; margin-top: 4px;">학습 에러: 높음<br>검증 에러: 높음</div>
      </div>
    </div>
    <div style="text-align: center; flex: 1; margin: 0 8px;">
      <div style="border: 3px solid #27ae60; border-radius: 8px; padding: 10px; background: #eafaf1; box-shadow: 0 2px 6px rgba(39,174,96,0.2);">
        <div style="font-weight: bold; color: #27ae60;">적절한 학습</div>
        <div style="font-size: 0.8em; color: #555; margin-top: 4px;">학습 에러: 낮음<br>검증 에러: 낮음</div>
      </div>
    </div>
    <div style="text-align: center; flex: 1;">
      <div style="border: 2px solid #e74c3c; border-radius: 8px; padding: 10px; background: #fdedec;">
        <div style="font-weight: bold; color: #e74c3c;">과적합</div>
        <div style="font-size: 0.8em; color: #555; margin-top: 4px;">학습 에러: 매우 낮음<br>검증 에러: 높음</div>
      </div>
    </div>
  </div>
  <div style="background: linear-gradient(to right, #3498db, #27ae60, #e74c3c); height: 8px; border-radius: 4px; margin: 4px 0;"></div>
  <div style="text-align: center; font-size: 0.85em; color: #7f8c8d; margin-top: 4px;">← 단순 &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 모델 복잡도 증가 &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 복잡 →</div>
</div>

> **핵심**: 학습 에러와 검증 에러의 차이(gap)가 과적합의 지표입니다. 이 차이가 작으면서 둘 다 낮은 지점이 이상적입니다.

## 1.3 K-Fold 교차 검증 (Cross-Validation)

### 왜 교차 검증이 필요한가?

데이터를 한 번만 train/validation으로 나누면, 나누는 방식에 따라 성능이 크게 달라질 수 있습니다.

**DevOps 비유: 부하 테스트**
- 한 번의 부하 테스트 결과만 보고 서버 성능을 판단하면 위험
- 여러 번, 다양한 조건에서 테스트해야 신뢰할 수 있는 결과를 얻을 수 있음
- K-Fold CV는 K번의 부하 테스트를 수행하는 것과 같음

### K-Fold CV 동작 방식 (K=5 기준)

<div style="margin: 20px 0; max-width: 600px;">
  <div style="font-weight: bold; margin-bottom: 10px;">전체 데이터를 5등분:</div>
  <table style="border-collapse: collapse; width: 100%; font-size: 0.9em;">
    <tr>
      <td style="padding: 6px 10px; font-weight: bold; color: #555;">Fold 1:</td>
      <td style="padding: 6px 8px; background: #fdedec; border: 1px solid #e74c3c; text-align: center; font-weight: bold; color: #e74c3c; border-radius: 4px;">검증</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 10px; color: #888;">→ Score 1</td>
    </tr>
    <tr>
      <td style="padding: 6px 10px; font-weight: bold; color: #555;">Fold 2:</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 8px; background: #fdedec; border: 1px solid #e74c3c; text-align: center; font-weight: bold; color: #e74c3c; border-radius: 4px;">검증</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 10px; color: #888;">→ Score 2</td>
    </tr>
    <tr>
      <td style="padding: 6px 10px; font-weight: bold; color: #555;">Fold 3:</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 8px; background: #fdedec; border: 1px solid #e74c3c; text-align: center; font-weight: bold; color: #e74c3c; border-radius: 4px;">검증</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 10px; color: #888;">→ Score 3</td>
    </tr>
    <tr>
      <td style="padding: 6px 10px; font-weight: bold; color: #555;">Fold 4:</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 8px; background: #fdedec; border: 1px solid #e74c3c; text-align: center; font-weight: bold; color: #e74c3c; border-radius: 4px;">검증</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 10px; color: #888;">→ Score 4</td>
    </tr>
    <tr>
      <td style="padding: 6px 10px; font-weight: bold; color: #555;">Fold 5:</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 8px; background: #ebf5fb; border: 1px solid #3498db; text-align: center; color: #2980b9;">학습</td>
      <td style="padding: 6px 8px; background: #fdedec; border: 1px solid #e74c3c; text-align: center; font-weight: bold; color: #e74c3c; border-radius: 4px;">검증</td>
      <td style="padding: 6px 10px; color: #888;">→ Score 5</td>
    </tr>
  </table>
  <div style="text-align: center; margin-top: 10px; padding: 8px; background: #eafaf1; border: 1.5px solid #27ae60; border-radius: 6px; font-weight: bold; color: #27ae60;">
    최종 CV Score = 5개 Score의 평균
  </div>
</div>

### 장점

1. **모든 데이터가 한 번씩 검증에 사용됨** - 데이터 낭비가 없음
2. **신뢰할 수 있는 성능 추정** - 단일 분할보다 안정적
3. **과적합 탐지** - Fold 간 점수 분산이 크면 과적합 의심
4. **OOF(Out-of-Fold) 예측** - 앙상블에 활용 가능

## 1.4 하이퍼파라미터

하이퍼파라미터는 모델이 학습하는 것이 아니라, **사람이 직접 설정하는 값**입니다.

**DevOps 비유: 시스템 설정값**
- `learning_rate` = 커널의 `swappiness` 값처럼, 학습의 공격성을 조절
- `max_depth` = Nginx의 `worker_connections`처럼, 모델의 최대 복잡도를 제한
- `num_leaves` = JVM의 `MaxHeapSize`처럼, 트리의 최대 잎 노드 수를 제한
- `n_estimators` = 카나리 배포의 단계 수처럼, 트리의 총 개수를 결정

### LightGBM 주요 하이퍼파라미터

| 파라미터 | 설명 | 일반적 범위 | 영향 |
|---------|------|-----------|------|
| `learning_rate` | 학습률 (각 트리의 기여도) | 0.01 ~ 0.1 | 작을수록 안정적이지만 느림 |
| `num_leaves` | 트리의 최대 잎 노드 수 | 7 ~ 127 | 클수록 복잡한 패턴 학습 가능하지만 과적합 위험 |
| `max_depth` | 트리의 최대 깊이 | 3 ~ 12 | 모델 복잡도 제한 |
| `min_child_samples` | 잎 노드의 최소 데이터 수 | 5 ~ 100 | 과적합 방지 |
| `subsample` | 학습에 사용할 데이터 비율 | 0.6 ~ 1.0 | 과적합 방지 (랜덤 샘플링) |
| `colsample_bytree` | 학습에 사용할 피처 비율 | 0.6 ~ 1.0 | 과적합 방지 (피처 샘플링) |
| `reg_alpha` | L1 정규화 | 0 ~ 10 | 희소한 피처 가중치를 0으로 |
| `reg_lambda` | L2 정규화 | 0 ~ 10 | 큰 가중치에 페널티 |

---

# Part 2: 실습 (60분)

## Step 1: 환경 설정 및 라이브러리 임포트

In [ ]:
# 필수 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 머신러닝 라이브러리
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

# 한글 깨짐 방지 (macOS 환경)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 시각화 설정
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
warnings.filterwarnings('ignore')

# 재현성을 위한 시드 고정
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('라이브러리 로딩 완료!')

## Step 2: 데이터 로딩 + 피처 엔지니어링

이전 세션(1~3회차)에서 배운 데이터 로딩 및 전처리 코드를 재활용합니다.

**DevOps 비유**: 이전에 만들어둔 Ansible Playbook을 재사용하는 것과 같습니다. 매번 처음부터 설정하지 않고 검증된 코드를 재활용합니다.

In [ ]:
def load_original_data():
    """원본 데이터를 로딩하고 기본 전처리를 수행합니다."""
    # 학습/테스트 데이터 로딩
    train = pd.read_csv('../input/train.csv')
    test = pd.read_csv('../input/test.csv')

    # 학습/테스트 구분을 위한 컬럼 추가
    train_copy = train.copy()
    train_copy['data'] = 'train'
    test_copy = test.copy()
    test_copy['data'] = 'test'
    test_copy['price'] = np.nan

    # 이상치 제거: 면적 대비 가격이 비정상적으로 낮은 데이터
    train_copy = train_copy[
        ~((train_copy['sqft_living'] > 12000) & (train_copy['price'] < 3000000))
    ].reset_index(drop=True)

    # 학습/테스트 데이터 합치기
    data = pd.concat([train_copy, test_copy], sort=False).reset_index(drop=True)
    data = data[train_copy.columns]

    # 날짜 컬럼 제거 (이번 실습에서는 사용하지 않음)
    data.drop('date', axis=1, inplace=True)

    # 우편번호를 문자열로 변환 (범주형 변수로 처리)
    data['zipcode'] = data['zipcode'].astype(str)

    # 타겟 변수(price)에 로그 변환 적용
    # 로그 변환: 큰 값의 영향력을 줄이고 정규분포에 가깝게 만듦
    data['price'] = np.log1p(data['price'])

    return data

print('데이터 로딩 함수 정의 완료')

In [ ]:
def train_test_split_custom(data, do_ohe=True):
    """전처리된 데이터를 학습/테스트로 분리합니다.
    
    Args:
        data: load_original_data()로 로딩된 데이터프레임
        do_ohe: True면 원-핫 인코딩, False면 레이블 인코딩
    
    Returns:
        X_train, X_test, y_train
    """
    # id, price, data 컬럼 제거 (피처로 사용하지 않는 컬럼)
    df = data.drop(['id', 'price', 'data'], axis=1).copy()

    # 범주형 변수 인코딩
    cat_cols = df.select_dtypes('object').columns
    for col in cat_cols:
        if do_ohe:
            # 원-핫 인코딩: 각 범주를 별도의 이진 컬럼으로 변환
            ohe_df = pd.get_dummies(df[[col]], prefix='ohe_' + col)
            df.drop(col, axis=1, inplace=True)
            df = pd.concat([df, ohe_df], axis=1)
        else:
            # 레이블 인코딩: 각 범주를 정수로 변환
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])

    # 학습/테스트 분리
    train_len = data[data['data'] == 'train'].shape[0]
    X_train = df.iloc[:train_len]
    X_test = df.iloc[train_len:]
    y_train = data[data['data'] == 'train']['price']

    return X_train, X_test, y_train

print('데이터 분리 함수 정의 완료')

In [ ]:
# 데이터 로딩 및 전처리 실행
data = load_original_data()
print(f'전체 데이터 크기: {data.shape}')
print(f'학습 데이터 수: {data[data["data"] == "train"].shape[0]}')
print(f'테스트 데이터 수: {data[data["data"] == "test"].shape[0]}')

In [ ]:
# LightGBM용 데이터 준비 (레이블 인코딩)
# LightGBM은 범주형 변수를 직접 다룰 수 있어 레이블 인코딩을 사용합니다
X_train_le, X_test_le, y_train = train_test_split_custom(data, do_ohe=False)
print(f'X_train 크기: {X_train_le.shape}')
print(f'X_test 크기: {X_test_le.shape}')
print(f'y_train 크기: {y_train.shape}')
print(f'\n피처 목록: {X_train_le.columns.tolist()}')

In [ ]:
# Ridge / RandomForest용 데이터 준비 (원-핫 인코딩)
# 선형 모델과 랜덤포레스트는 원-핫 인코딩이 더 적합합니다
X_train_ohe, X_test_ohe, _ = train_test_split_custom(data, do_ohe=True)
print(f'원-핫 인코딩 후 X_train 크기: {X_train_ohe.shape}')
print(f'원-핫 인코딩 후 X_test 크기: {X_test_ohe.shape}')

## Step 3: 평가 지표 및 CV 함수 정의

### 평가 지표: RMSE (Root Mean Squared Error)

이 대회에서는 실제 가격 스케일의 RMSE를 사용합니다. 우리는 `log1p(price)`를 타겟으로 학습하므로, 평가 시에는 `expm1()`으로 원래 스케일로 역변환한 뒤 RMSE를 계산해야 합니다.

**DevOps 비유**: SLA 측정 시 응답시간을 로그 스케일로 모니터링하다가, 보고서에는 원래 밀리초 단위로 변환해서 보여주는 것과 같습니다.

In [ ]:
def rmse_exp(y_true, y_pred):
    """로그 스케일의 예측값을 원래 스케일로 변환한 후 RMSE를 계산합니다.
    
    Args:
        y_true: 실제값 (로그 스케일)
        y_pred: 예측값 (로그 스케일)
    
    Returns:
        RMSE (원래 가격 스케일)
    """
    return np.sqrt(mean_squared_error(np.expm1(y_true), np.expm1(y_pred)))

print('평가 함수 정의 완료')

### LightGBM 5-Fold CV 함수

아래 함수는 LightGBM을 5-Fold 교차 검증으로 학습하는 핵심 함수입니다.

**이 함수가 하는 일:**
1. 데이터를 5등분으로 나눔
2. 각 Fold에서 4/5로 학습, 1/5로 검증
3. OOF(Out-of-Fold) 예측값 생성
4. 테스트 데이터 예측값을 5개 Fold의 평균으로 계산
5. 피처 중요도(Feature Importance) 기록

**DevOps 비유**: 카나리 배포를 5번 수행하면서 각 배포의 성능을 측정하고, 최종 결과는 5번의 평균을 사용하는 것과 같습니다.

In [ ]:
def get_oof_lgb(X_train, y_train, X_test, lgb_param, verbose_eval=False):
    """LightGBM 5-Fold CV를 수행하고 OOF 예측값을 반환합니다.
    
    Args:
        X_train: 학습 피처 데이터프레임
        y_train: 학습 타겟 시리즈
        X_test: 테스트 피처 데이터프레임
        lgb_param: LightGBM 하이퍼파라미터 딕셔너리
        verbose_eval: 학습 과정 출력 간격 (False면 출력 안 함)
    
    Returns:
        oof: OOF 예측값 (학습 데이터에 대한 검증 예측값)
        predictions: 테스트 데이터에 대한 평균 예측값
        cv_score: CV 점수 (RMSE)
        feature_importance_df: 피처 중요도 데이터프레임
    """
    # 5-Fold 분할 설정
    folds = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

    # OOF 예측값과 테스트 예측값을 저장할 배열 초기화
    oof = np.zeros(len(X_train))
    predictions = np.zeros(len(X_test))

    # 피처 중요도를 저장할 데이터프레임
    feature_importance_df = pd.DataFrame()

    # LightGBM 4.x 콜백 설정
    callbacks = [lgb.early_stopping(200)]
    if verbose_eval and verbose_eval > 0:
        callbacks.append(lgb.log_evaluation(verbose_eval))
    else:
        callbacks.append(lgb.log_evaluation(-1))

    # 각 Fold별로 학습 및 예측 수행
    for fold_, (trn_idx, val_idx) in enumerate(folds.split(X_train.values, y_train.values)):
        if verbose_eval and verbose_eval > 0:
            print(f'Fold : {fold_ + 1}')

        # LightGBM 데이터셋 생성
        trn_data = lgb.Dataset(X_train.iloc[trn_idx], label=y_train.iloc[trn_idx])
        val_data = lgb.Dataset(X_train.iloc[val_idx], label=y_train.iloc[val_idx])

        # 모델 학습 (조기 종료 적용)
        clf = lgb.train(
            lgb_param,
            trn_data,
            100000,  # 최대 반복 횟수
            valid_sets=[trn_data, val_data],
            callbacks=callbacks
        )

        # 검증 데이터에 대한 예측 (OOF)
        oof[val_idx] = clf.predict(X_train.iloc[val_idx], num_iteration=clf.best_iteration)

        # 테스트 데이터에 대한 예측 (5개 Fold의 평균)
        predictions += clf.predict(X_test, num_iteration=clf.best_iteration) / folds.n_splits

        # 피처 중요도 기록
        fold_importance_df = pd.DataFrame()
        fold_importance_df['feature'] = X_train.columns.tolist()
        fold_importance_df['importance'] = clf.feature_importance('gain')
        fold_importance_df['fold'] = fold_ + 1
        feature_importance_df = pd.concat([feature_importance_df, fold_importance_df], axis=0)

    # 전체 CV 점수 계산
    cv_score = rmse_exp(y_train, oof)
    print(f'CV-Score: {cv_score:.6f}')

    return oof, predictions, cv_score, feature_importance_df

print('LightGBM CV 함수 정의 완료')

## Step 4: LightGBM 기본 모델 학습

먼저 기본 하이퍼파라미터로 LightGBM 모델을 학습해보겠습니다. 이 결과를 기준(baseline)으로 삼고, 이후 하이퍼파라미터를 변경하며 성능 변화를 관찰할 것입니다.

**DevOps 비유**: 기본 설정으로 서버를 먼저 구동한 뒤, 성능 테스트 결과를 보면서 튜닝하는 것과 같습니다.

In [ ]:
# LightGBM 기본 하이퍼파라미터 설정
lgb_param_baseline = {
    'objective': 'regression',     # 회귀 문제
    'metric': 'rmse',              # 평가 지표: RMSE
    'boosting_type': 'gbdt',       # 부스팅 방식: Gradient Boosted Decision Trees
    'learning_rate': 0.05,         # 학습률
    'num_leaves': 15,              # 트리의 최대 잎 노드 수
    'max_depth': -1,               # 트리 깊이 제한 없음
    'min_child_samples': 20,       # 잎 노드의 최소 데이터 수
    'subsample': 0.8,              # 데이터 샘플링 비율
    'colsample_bytree': 0.8,       # 피처 샘플링 비율
    'reg_alpha': 0.1,              # L1 정규화
    'reg_lambda': 0.1,             # L2 정규화
    'random_state': RANDOM_SEED,
    'n_jobs': -1,                  # 모든 CPU 코어 사용
    'verbose': -1                  # 학습 로그 출력 안 함
}

print('기본 하이퍼파라미터 설정:')
for key, value in lgb_param_baseline.items():
    print(f'  {key}: {value}')

In [ ]:
# 기본 파라미터로 5-Fold CV 수행
print('=' * 60)
print('LightGBM Baseline 모델 학습 시작')
print('=' * 60)

oof_baseline, pred_baseline, score_baseline, fi_baseline = get_oof_lgb(
    X_train_le, y_train, X_test_le, lgb_param_baseline, verbose_eval=500
)

print(f'\nBaseline CV-Score: {score_baseline:.6f}')

## Step 5: Feature Importance 시각화

모델이 어떤 피처를 중요하게 사용했는지 확인합니다. 이를 통해 도메인 지식과 모델의 판단이 일치하는지 검증할 수 있습니다.

**DevOps 비유**: 장애 분석에서 Root Cause Analysis를 하듯이, 모델의 예측에 가장 큰 영향을 미친 요인을 찾는 과정입니다.

In [ ]:
def plot_feature_importance(fi_df, num_feature=20):
    """피처 중요도를 시각화합니다.
    
    Args:
        fi_df: get_oof_lgb()에서 반환된 피처 중요도 데이터프레임
        num_feature: 표시할 상위 피처 개수
    """
    # 5개 Fold의 평균 중요도 기준으로 상위 피처 선택
    cols = (
        fi_df[['feature', 'importance']]
        .groupby('feature')
        .mean()
        .sort_values(by='importance', ascending=False)[:num_feature]
        .index
    )

    best_features = fi_df.loc[fi_df.feature.isin(cols)]

    # 시각화
    plt.figure(figsize=(10, 8))
    sns.barplot(
        x='importance',
        y='feature',
        data=best_features.sort_values(by='importance', ascending=False)
    )
    plt.title('Feature Importances (averaged over folds)')
    plt.tight_layout()
    plt.show()

# 피처 중요도 시각화
plot_feature_importance(fi_baseline)

### 피처 중요도 해석

위 그래프에서 확인할 수 있는 점:
- **높은 중요도 피처**: 집의 크기(sqft_living), 위치(lat, long), 등급(grade) 등이 가격 예측에 가장 중요
- **낮은 중요도 피처**: 모델이 거의 사용하지 않는 피처는 제거를 검토할 수 있음

> **TIP**: 도메인 지식과 모델의 판단이 일치하는지 확인하는 것이 중요합니다. 만약 의미 없는 피처가 상위에 있다면 과적합을 의심해볼 수 있습니다.

## Step 6: 하이퍼파라미터 실험

이제 핵심 하이퍼파라미터를 변경하면서 CV Score의 변화를 관찰합니다.

**DevOps 비유**: 서버 성능 튜닝에서 한 번에 하나의 파라미터만 변경하면서 성능 변화를 측정하는 것과 같습니다. 동시에 여러 값을 바꾸면 어떤 변경이 성능에 영향을 미쳤는지 알 수 없습니다.

### 실험 1: `num_leaves` 변경 (7 vs 15 vs 63)

`num_leaves`는 트리의 최대 잎 노드 수를 결정합니다.
- **작은 값 (7)**: 단순한 트리 -> 과소적합 위험
- **중간 값 (15)**: 적절한 복잡도 (기본값)
- **큰 값 (63)**: 복잡한 트리 -> 과적합 위험

In [ ]:
# num_leaves 실험
num_leaves_values = [7, 15, 63]
results_num_leaves = {}

for nl in num_leaves_values:
    print(f'\n{"=" * 60}')
    print(f'num_leaves = {nl}')
    print(f'{"=" * 60}')

    # 기본 파라미터를 복사하고 num_leaves만 변경
    param = lgb_param_baseline.copy()
    param['num_leaves'] = nl

    oof, pred, score, fi = get_oof_lgb(
        X_train_le, y_train, X_test_le, param, verbose_eval=False
    )
    results_num_leaves[nl] = score

# 결과 비교
print(f'\n{"=" * 60}')
print('num_leaves 실험 결과 요약')
print(f'{"=" * 60}')
for nl, score in results_num_leaves.items():
    marker = ' <-- baseline' if nl == 15 else ''
    print(f'  num_leaves={nl:3d}  |  CV-Score: {score:.6f}{marker}')

### 실험 2: `learning_rate` 변경 (0.01 vs 0.05 vs 0.1)

`learning_rate`는 각 트리가 전체 모델에 기여하는 정도를 조절합니다.
- **작은 값 (0.01)**: 느리지만 세밀한 학습 -> 트리를 많이 필요로 함
- **중간 값 (0.05)**: 적절한 균형 (기본값)
- **큰 값 (0.1)**: 빠르지만 거친 학습 -> 과적합 위험

**DevOps 비유**: 카나리 배포에서 트래픽 전환 비율과 같습니다.
- 1%씩 전환(작은 lr): 안전하지만 전체 배포까지 오래 걸림
- 5%씩 전환(중간 lr): 적절한 속도와 안전성
- 10%씩 전환(큰 lr): 빠르지만 문제 발생 시 영향 범위가 큼

In [ ]:
# learning_rate 실험
lr_values = [0.01, 0.05, 0.1]
results_lr = {}

for lr in lr_values:
    print(f'\n{"=" * 60}')
    print(f'learning_rate = {lr}')
    print(f'{"=" * 60}')

    # 기본 파라미터를 복사하고 learning_rate만 변경
    param = lgb_param_baseline.copy()
    param['learning_rate'] = lr

    oof, pred, score, fi = get_oof_lgb(
        X_train_le, y_train, X_test_le, param, verbose_eval=False
    )
    results_lr[lr] = score

# 결과 비교
print(f'\n{"=" * 60}')
print('learning_rate 실험 결과 요약')
print(f'{"=" * 60}')
for lr, score in results_lr.items():
    marker = ' <-- baseline' if lr == 0.05 else ''
    print(f'  learning_rate={lr:.2f}  |  CV-Score: {score:.6f}{marker}')

### 하이퍼파라미터 실험 결과 종합

실험 결과를 한눈에 비교합니다. CV-Score가 낮을수록(RMSE이므로) 성능이 좋은 것입니다.

In [ ]:
# 실험 결과 종합 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# num_leaves 실험 결과
axes[0].bar(
    [str(k) for k in results_num_leaves.keys()],
    results_num_leaves.values(),
    color=['#3498db', '#2ecc71', '#e74c3c']
)
axes[0].set_title('num_leaves vs CV-Score')
axes[0].set_xlabel('num_leaves')
axes[0].set_ylabel('CV-Score (RMSE, lower is better)')
for i, (k, v) in enumerate(results_num_leaves.items()):
    axes[0].text(i, v, f'{v:.0f}', ha='center', va='bottom', fontweight='bold')

# learning_rate 실험 결과
axes[1].bar(
    [str(k) for k in results_lr.keys()],
    results_lr.values(),
    color=['#3498db', '#2ecc71', '#e74c3c']
)
axes[1].set_title('learning_rate vs CV-Score')
axes[1].set_xlabel('learning_rate')
axes[1].set_ylabel('CV-Score (RMSE, lower is better)')
for i, (k, v) in enumerate(results_lr.items()):
    axes[1].text(i, v, f'{v:.0f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## Step 7: 다양한 모델 간단 비교 (Ridge vs RandomForest vs LightGBM)

이론에서 배운 세 가지 모델 계열을 직접 비교해봅니다. 동일한 K-Fold CV 설정에서 각 모델의 성능을 측정합니다.

**DevOps 비유**: 동일한 워크로드에서 Apache vs Nginx vs Caddy의 성능을 벤치마크하는 것과 같습니다.

In [ ]:
def cv_sklearn_model(model, X_train, y_train, model_name='Model'):
    """sklearn 모델에 대해 5-Fold CV를 수행합니다.
    
    Args:
        model: sklearn 모델 인스턴스
        X_train: 학습 피처 데이터프레임
        y_train: 학습 타겟 시리즈
        model_name: 출력용 모델 이름
    
    Returns:
        cv_score: 평균 CV 점수 (RMSE)
        fold_scores: 각 Fold별 점수 리스트
    """
    folds = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    oof = np.zeros(len(X_train))
    fold_scores = []

    for fold_, (trn_idx, val_idx) in enumerate(folds.split(X_train.values, y_train.values)):
        # 학습/검증 분리
        X_trn, X_val = X_train.iloc[trn_idx], X_train.iloc[val_idx]
        y_trn, y_val = y_train.iloc[trn_idx], y_train.iloc[val_idx]

        # 모델 학습
        model.fit(X_trn, y_trn)

        # 검증 예측
        oof[val_idx] = model.predict(X_val)

        # Fold별 점수 계산
        fold_score = rmse_exp(y_val, oof[val_idx])
        fold_scores.append(fold_score)
        print(f'  Fold {fold_ + 1}: {fold_score:.6f}')

    # 전체 CV 점수
    cv_score = rmse_exp(y_train, oof)
    print(f'  {model_name} CV-Score: {cv_score:.6f}')
    print(f'  Fold 점수 표준편차: {np.std(fold_scores):.6f}')

    return cv_score, fold_scores

print('sklearn CV 함수 정의 완료')

In [ ]:
# 모델 비교 결과를 저장할 딕셔너리
model_comparison = {}

# 1) Ridge Regression (선형 모델)
print('=' * 60)
print('1. Ridge Regression (선형 모델)')
print('=' * 60)
ridge = Ridge(alpha=10.0, random_state=RANDOM_SEED)
score_ridge, folds_ridge = cv_sklearn_model(
    ridge, X_train_ohe, y_train, 'Ridge'
)
model_comparison['Ridge'] = score_ridge

In [ ]:
# 2) Random Forest (트리 기반 모델)
print('=' * 60)
print('2. Random Forest (트리 기반 모델)')
print('=' * 60)
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
score_rf, folds_rf = cv_sklearn_model(
    rf, X_train_ohe, y_train, 'RandomForest'
)
model_comparison['RandomForest'] = score_rf

In [ ]:
# 3) LightGBM (부스팅 모델) - 이미 학습한 baseline 결과 사용
print('=' * 60)
print('3. LightGBM (부스팅 모델)')
print('=' * 60)
print(f'  LightGBM CV-Score: {score_baseline:.6f} (baseline 결과 재사용)')
model_comparison['LightGBM'] = score_baseline

In [ ]:
# 모델 비교 시각화
plt.figure(figsize=(10, 6))

models = list(model_comparison.keys())
scores = list(model_comparison.values())
colors = ['#3498db', '#2ecc71', '#e74c3c']

bars = plt.bar(models, scores, color=colors, edgecolor='black', linewidth=0.5)

# 각 막대 위에 점수 표시
for bar, score in zip(bars, scores):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f'{score:,.0f}',
        ha='center', va='bottom',
        fontweight='bold', fontsize=12
    )

plt.title('Model Comparison (CV-Score, RMSE - lower is better)', fontsize=14)
plt.ylabel('CV-Score (RMSE)')
plt.tight_layout()
plt.show()

# 최고 성능 모델 출력
best_model = min(model_comparison, key=model_comparison.get)
print(f'\n최고 성능 모델: {best_model} (CV-Score: {model_comparison[best_model]:,.6f})')

### 모델 비교 정리

| 모델 | 특징 | 장점 | 단점 |
|------|------|------|------|
| **Ridge** | 선형 모델 | 빠르고 해석 쉬움 | 비선형 패턴 포착 불가 |
| **RandomForest** | 트리 앙상블 | 과적합에 강함 | 느리고 메모리 많이 사용 |
| **LightGBM** | 부스팅 | 높은 성능, 빠른 학습 | 하이퍼파라미터 튜닝 필요 |

> **결론**: 정형 데이터에서는 대부분 LightGBM 같은 부스팅 모델이 가장 좋은 성능을 보입니다. 실무에서도 LightGBM 또는 XGBoost가 가장 많이 사용됩니다.

## Step 8: 최종 제출 파일 생성

가장 성능이 좋았던 LightGBM baseline 모델의 예측 결과를 제출 파일로 만듭니다.

**중요**: 학습 시 `log1p(price)`를 타겟으로 사용했으므로, 제출 시에는 `expm1()`으로 원래 가격 스케일로 역변환해야 합니다.

<div style="display: flex; align-items: center; gap: 6px; flex-wrap: wrap; justify-content: center; margin: 15px 0;">
  <div style="border: 2px solid #3498db; border-radius: 6px; padding: 8px 14px; background: #ebf5fb; text-align: center; font-weight: bold;">원본 가격</div>
  <div style="text-align: center;">
    <div style="font-size: 0.75em; color: #8e44ad; font-weight: bold;">log1p</div>
    <div style="font-size: 1.3em; color: #7f8c8d;">→</div>
  </div>
  <div style="border: 2px solid #8e44ad; border-radius: 6px; padding: 8px 14px; background: #f4ecf7; text-align: center; font-weight: bold;">로그 스케일 학습</div>
  <div style="text-align: center;">
    <div style="font-size: 0.75em; color: #27ae60; font-weight: bold;">expm1</div>
    <div style="font-size: 1.3em; color: #7f8c8d;">→</div>
  </div>
  <div style="border: 2px solid #27ae60; border-radius: 6px; padding: 8px 14px; background: #eafaf1; text-align: center; font-weight: bold;">원본 가격 복원</div>
</div>

In [ ]:
# 테스트 데이터의 id 가져오기
test_original = pd.read_csv('../input/test.csv')

# 제출 파일 생성
submission = pd.DataFrame({
    'id': test_original['id'],
    'price': np.expm1(pred_baseline)  # 로그 역변환
})

# 결과 확인
print('제출 파일 미리보기:')
print(submission.head(10))
print(f'\n예측 가격 통계:')
print(submission['price'].describe())

In [ ]:
# CSV 파일로 저장
submission.to_csv('submission.csv', index=False)
print('submission.csv 파일이 생성되었습니다!')
print(f'파일 크기: {submission.shape[0]}행 x {submission.shape[1]}열')

In [ ]:
# 예측 가격 분포 확인 (정상 범위인지 검증)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 학습 데이터의 실제 가격 분포
train_prices = np.expm1(y_train)
axes[0].hist(train_prices, bins=50, color='#3498db', alpha=0.7, edgecolor='black')
axes[0].set_title('Train Data - Actual Price Distribution')
axes[0].set_xlabel('Price')
axes[0].set_ylabel('Count')

# 테스트 데이터의 예측 가격 분포
axes[1].hist(submission['price'], bins=50, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[1].set_title('Test Data - Predicted Price Distribution')
axes[1].set_xlabel('Price')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print('학습 데이터와 테스트 데이터의 가격 분포가 비슷하면 정상입니다.')

---

# 연습 문제

아래 연습 문제를 풀어보면서 오늘 배운 내용을 복습해봅시다.

## 연습 1: `subsample`과 `colsample_bytree` 실험

데이터와 피처의 샘플링 비율을 조절하여 과적합을 방지하는 실험을 수행해보세요.

- `subsample`: 0.6 vs 0.8 vs 1.0
- 각 값에 대해 CV Score를 비교하고 결과를 해석해보세요.

**힌트**: `lgb_param_baseline`을 복사한 뒤 `subsample` 값만 변경하면 됩니다.

In [ ]:
# 여기에 코드를 작성하세요
# subsample_values = [0.6, 0.8, 1.0]
# ...


## 연습 2: Ridge의 `alpha` 파라미터 실험

Ridge Regression의 정규화 강도(`alpha`)를 변경하며 성능 변화를 관찰해보세요.

- `alpha` 값: 0.1, 1.0, 10.0, 100.0
- alpha가 커질수록 정규화가 강해져서 가중치가 작아집니다.
- 위에서 정의한 `cv_sklearn_model()` 함수를 사용하세요.

**힌트**: `Ridge(alpha=값)` 으로 모델을 생성하고, `X_train_ohe`를 입력으로 사용하세요.

In [ ]:
# 여기에 코드를 작성하세요
# alpha_values = [0.1, 1.0, 10.0, 100.0]
# ...


## 연습 3: 간단한 피처 엔지니어링 추가 후 성능 비교

다음 피처를 추가한 뒤 LightGBM CV Score가 개선되는지 확인해보세요.

추가할 피처:
- `sqft_living_ratio`: 전체 면적 대비 거실 면적 비율 (`sqft_living / sqft_lot`)
- `total_rooms`: 총 방 수 (`bedrooms + bathrooms`)
- `is_renovated`: 리모델링 여부 (`yr_renovated > 0`이면 1, 아니면 0)

**힌트**: `load_original_data()` 결과에 피처를 추가한 뒤, `train_test_split_custom()`으로 다시 분리하세요.

In [ ]:
# 여기에 코드를 작성하세요
# data_v2 = load_original_data()
# data_v2['sqft_living_ratio'] = ...
# data_v2['total_rooms'] = ...
# data_v2['is_renovated'] = ...
# X_train_v2, X_test_v2, y_train_v2 = train_test_split_custom(data_v2, do_ohe=False)
# ...


## 연습 4: K 값 변경 실험 (3-Fold vs 5-Fold vs 10-Fold)

`get_oof_lgb()` 함수의 `KFold(n_splits=5)`를 수정하여 K 값에 따른 CV Score 변화를 관찰해보세요.

- K=3: 빠르지만 각 Fold의 학습 데이터가 적어 불안정
- K=5: 일반적으로 가장 많이 사용되는 설정 (기본값)
- K=10: 안정적이지만 10번 학습해야 해서 느림

**생각해볼 점**: K가 커지면 어떤 장단점이 있을까요? DevOps에서 A/B 테스트의 트래픽 분할 비율을 바꾸는 것과 어떤 유사점이 있을까요?

**힌트**: `get_oof_lgb()` 함수를 복사하여 `n_splits` 파라미터를 추가하거나, 함수 내부의 값을 직접 변경해보세요.

In [ ]:
# 여기에 코드를 작성하세요
# def get_oof_lgb_kfold(X_train, y_train, X_test, lgb_param, n_splits=5, verbose_eval=False):
#     folds = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
#     ...


---

## 오늘의 요약

### 핵심 개념

1. **모델 선택**: 정형 데이터에서는 LightGBM 같은 부스팅 모델이 일반적으로 가장 좋은 성능을 보임
2. **과적합 방지**: 적절한 하이퍼파라미터 설정과 교차 검증을 통해 과적합을 탐지하고 방지
3. **교차 검증**: K-Fold CV를 통해 신뢰할 수 있는 성능 측정이 가능
4. **하이퍼파라미터 튜닝**: 한 번에 하나씩 변경하며 체계적으로 실험
5. **피처 중요도**: 모델이 어떤 피처를 중요하게 사용하는지 확인하여 도메인 지식 검증

### DevOps 관점에서의 정리

| ML 개념 | DevOps 비유 |
|---------|------------|
| K-Fold CV | 여러 번의 부하 테스트 |
| 하이퍼파라미터 | 커널/JVM 설정값 |
| 과적합 | 알람이 너무 민감 |
| 과소적합 | 알람이 너무 둔감 |
| Feature Importance | Root Cause Analysis |
| 모델 비교 | 서버 벤치마크 |